## mode_0008.xsf 클린(벡터 크기 기준)

- `PRIMCOORD` 아래의 각 원자 라인에 있는 변위벡터 `(dx, dy, dz)`의 **모듈러스** `sqrt(dx^2+dy^2+dz^2)`를 계산합니다.
- 모듈러스가 `cutoff` 이하이면 해당 원자의 변위벡터를 `(0, 0, 0)`으로 만들어 `*_clean.xsf`로 저장합니다.

> 참고: “원자 줄 자체를 삭제(원자 수 감소)”가 아니라, **벡터만 0으로 만들어** 남깁니다(모드/구조 포맷 호환성 목적).


In [2]:
import os
import math

# =====================
# 설정
# =====================
input_path = "mode_0008.xsf"  # 예: "mode_0008.xsf" (같은 폴더에 두고 실행)
cutoff = 0.1                # 이 값 이하 모듈러스(벡터 크기) -> (0,0,0)

# =====================
# 처리
# =====================

def clean_xsf(input_path: str, cutoff: float) -> str:
    with open(input_path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    # PRIMCOORD 위치 찾기
    primcoord_idx = None
    for i, line in enumerate(lines):
        if line.strip() == "PRIMCOORD":
            primcoord_idx = i
            break
    if primcoord_idx is None:
        raise ValueError("PRIMCOORD 블록을 찾지 못했습니다.")
    if primcoord_idx + 2 >= len(lines):
        raise ValueError("PRIMCOORD 헤더/원자 라인이 충분하지 않습니다.")

    # 원자 수/모드 번호 라인 (예: "41 1")
    header_tokens = lines[primcoord_idx + 1].split()
    if len(header_tokens) < 2:
        raise ValueError("PRIMCOORD 헤더 라인이 'N M' 형태가 아닙니다: " + lines[primcoord_idx + 1].strip())
    n_atoms = int(header_tokens[0])

    atom_start = primcoord_idx + 2
    atom_end = atom_start + n_atoms
    if atom_end > len(lines):
        raise ValueError(f"예상 원자 수({n_atoms})만큼 라인이 없습니다. 현재 lines={len(lines)}")

    kept = 0

    new_lines = list(lines)

    for j in range(n_atoms):
        line_idx = atom_start + j
        toks = new_lines[line_idx].split()
        if len(toks) < 7:
            raise ValueError(f"원자 라인 토큰 수가 7 미만입니다: line={new_lines[line_idx].rstrip()}" )

        elem = toks[0]
        # x y z dx dy dz
        x, y, z, dx, dy, dz = map(float, toks[1:7])
        mod = math.sqrt(dx*dx + dy*dy + dz*dz)

        if mod <= cutoff:
            dx2 = dy2 = dz2 = 0.0
        else:
            dx2, dy2, dz2 = dx, dy, dz
            kept += 1

        # 포맷은 유연하지만, float를 고정밀로 출력
        new_lines[line_idx] = (
            f"{elem:2s} "
            f"{x: .16f} {y: .16f} {z: .16f} "
            f"{dx2: .16f} {dy2: .16f} {dz2: .16f}\n"
        )

    out_path = input_path
    if out_path.endswith(".xsf"):
        out_path = out_path[:-4] + "_clean.xsf"
    else:
        out_path = out_path + "_clean"

    with open(out_path, "w", encoding="utf-8") as f:
        f.writelines(new_lines)

    print(f"입력: {input_path}")
    print(f"출력: {out_path}")
    print(f"cutoff: {cutoff}")
    print(f"모듈러스 > cutoff 인 원자(벡터 유지) 수: {kept} / {n_atoms}")

    return out_path


# 실행
clean_xsf(input_path, cutoff)


입력: mode_0008.xsf
출력: mode_0008_clean.xsf
cutoff: 0.1
모듈러스 > cutoff 인 원자(벡터 유지) 수: 2 / 41


'mode_0008_clean.xsf'